# BeeWatch AI — Confronto fra modelli

**Task M4-T3** · Tre Pipeline scikit-learn (preprocessore di M4-T2 + regressore) confrontate con le quattro baseline di M4-T1.

Questo notebook **non contiene logica di training**: richiama `beewatch/ml/train.py`, dove stanno split, feature, modelli e metriche, coperti da test. Qui si guardano i risultati.

Regole del confronto (DA-05):

- dati: pannello **Q8** letto da MySQL (40 stati × 15 anni), mai dal CSV;
- addestramento 1998–2009, verifica **2010–2012** (120 righe), le stesse per modelli e baseline;
- stessa funzione `metriche()` per MAE, RMSE e R²;
- un modello è accettabile solo se `MAE_modello < MAE_persistenza`, sui valori **non arrotondati**.

Serve un database con i dati USDA caricati (`python scripts/run_etl.py`).

In [1]:
import pandas as pd

from beewatch.config import ottieni
from beewatch.database import Database
from beewatch.ml.train import (
    PERSISTENZA,
    RANDOM_FOREST,
    SEED,
    carica_dataset_training,
    confronta_modelli,
    soddisfa_criterio,
)

pd.options.display.float_format = "{:.4f}".format

dati = carica_dataset_training(Database(ottieni().database))
stati, anni = dati["codice_stato"].nunique(), (dati["anno"].min(), dati["anno"].max())
print(f"{len(dati)} righe, {stati} stati, anni {anni[0]}-{anni[1]}")

600 righe, 40 stati, anni 1998-2012


## 1 · Modelli e baseline sulla stessa verifica

Seed fissato (`SEED = 42`): lo stesso confronto, rilanciato, dà gli stessi numeri. Gli iperparametri sono quelli di default: il tuning è di M4-T5.

In [2]:
confronto = confronta_modelli(dati, seed=SEED)
risultati = confronto.risultati()
baseline = {n: m for n, m in risultati.items() if n not in confronto.modelli}

tabella = pd.DataFrame(
    {
        "tipo": ["modello" if n in confronto.modelli else "baseline" for n in risultati],
        "MAE": [m.mae_kg for m in risultati.values()],
        "RMSE": [m.rmse_kg for m in risultati.values()],
        "R²": [m.r2 for m in risultati.values()],
        "batte la persistenza (DA-05)": [
            soddisfa_criterio(m, baseline) if n in confronto.modelli else None
            for n, m in risultati.items()
        ],
    },
    index=list(risultati),
)
anni_verifica = sorted(confronto.verifica["anno"].unique().tolist())
print("righe di verifica:", len(confronto.verifica), "· anni:", anni_verifica)
tabella.sort_values("MAE")

righe di verifica: 120 · anni: [2010, 2011, 2012]


,tipo,MAE,RMSE,R²,batte la persistenza (DA-05)
random forest,modello,3.6558,4.5698,0.6088,True
regressione lineare,modello,4.0254,5.3358,0.4666,True
persistenza (resa dell'anno prima),baseline,4.0634,5.2964,0.4745,None
media dello stato + tendenza annuale,baseline,4.1681,5.4350,0.4466,None
albero di decisione,modello,5.5792,6.8581,0.1189,False
media dello stato,baseline,5.7272,6.7982,0.1342,None
media globale,baseline,6.6546,8.3711,-0.3128,None


## 2 · Il criterio DA-05, a piena precisione

In [3]:
mae_persistenza = risultati[PERSISTENZA].mae_kg
print(f"persistenza: MAE = {mae_persistenza!r}")
for nome in confronto.modelli:
    mae = risultati[nome].mae_kg
    print(
        f"{nome:22s} MAE = {mae!r:22}  differenza = {mae - mae_persistenza:+.6f}  ->  "
        f"{'soddisfa' if soddisfa_criterio(risultati[nome], baseline) else 'NON soddisfa'} DA-05"
    )
print("vincitore:", confronto.vincitore())

persistenza: MAE = 4.0634283333333325
regressione lineare    MAE = 4.025411864980751       differenza = -0.038016  ->  soddisfa DA-05
albero di decisione    MAE = 5.579183333333333       differenza = +1.515755  ->  NON soddisfa DA-05
random forest          MAE = 3.6558007499999965      differenza = -0.407628  ->  soddisfa DA-05
vincitore: random forest


## 3 · Anno per anno

Il criterio si decide sulle 120 righe insieme. Separare gli anni mostra quanto il risultato è stabile: con 40 righe per anno la differenza fra due metodi può cambiare segno.

In [4]:
per_anno = confronto.risultati_per_anno()
mae_per_anno = pd.DataFrame(
    {n: {a: m.mae_kg for a, m in anni.items()} for n, anni in per_anno.items()}
).T
mae_per_anno["media"] = mae_per_anno[[2010, 2011, 2012]].mean(axis=1)
mae_per_anno["dev. std"] = mae_per_anno[[2010, 2011, 2012]].std(axis=1)
mae_per_anno

,2010,2011,2012,media,dev. std
regressione lineare,4.4181,3.9638,3.6944,4.0254,0.3658
albero di decisione,5.1823,5.1596,6.3956,5.5792,0.7072
random forest,3.4992,3.3866,4.0815,3.6558,0.3730
media globale,6.6773,6.8700,6.4165,6.6546,0.2276
media dello stato,5.6964,6.0668,5.4185,5.7272,0.3252
media dello stato + tendenza annuale,4.5678,4.0487,3.8878,4.1681,0.3554
persistenza (resa dell'anno prima),4.0483,4.6040,3.5380,4.0634,0.5331


## 4 · Quanto dipende dal seed

Informativo, **non** usato per scegliere: il risultato ufficiale resta quello con `SEED = 42`, fissato prima del confronto. Albero e foresta sono casuali; la regressione lineare no.

In [5]:
semi = range(10)
mae_per_seme = pd.DataFrame(
    {s: {n: m.mae_kg for n, m in confronta_modelli(dati, seed=s).risultati().items()} for s in semi}
).loc[list(confronto.modelli)]
pd.DataFrame({"minimo": mae_per_seme.min(axis=1), "massimo": mae_per_seme.max(axis=1)})

,minimo,massimo
regressione lineare,4.0254,4.0254
albero di decisione,5.2125,5.5111
random forest,3.6131,3.7779


In [6]:
print(
    "random forest sotto la persistenza con tutti i seed:",
    bool((mae_per_seme.loc[RANDOM_FOREST] < mae_persistenza).all()),
)

random forest sotto la persistenza con tutti i seed: True


## Conclusioni

Le conclusioni e la scelta motivata sono nella [model card](../docs/model_card.md), sezione *Confronto fra modelli*. In breve: random forest e regressione lineare battono la persistenza sul MAE dell'intera verifica, l'albero di decisione no; vince la random forest. Il margine della regressione lineare è piccolo, e nel 2012 la persistenza fa meglio di tutti i modelli. La verifica è stata usata anche per scegliere: la cross-validation temporale di M4-T4 dirà se il vantaggio regge su altri anni.